In [ ]:
import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import matplotlib.pyplot as plt

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA_DIR = Path("./statoil-iceberg-classifier-challenge/data/statoil-iceberg-classifier-challenge")

TRAIN_PATH = DATA_DIR / "train.json/train.json"
TEST_PATH = DATA_DIR / "test.json/test.json"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv/sample_submission.csv"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CFG = {
    "img_size": 75,
    "channels": 3,
    "batch_size": 64,
    "epochs": 35,
    "n_folds": 5,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "num_workers": 2,
    "patience": 8,
    "tta": True,
}

print("Train exists:", TRAIN_PATH.exists(), TRAIN_PATH)
print("Test exists:", TEST_PATH.exists(), TEST_PATH)
print("Sample exists:", SAMPLE_PATH.exists(), SAMPLE_PATH)

In [ ]:
train_df = pd.read_json(TRAIN_PATH)
test_df = pd.read_json(TEST_PATH)

print(train_df.shape, test_df.shape)
train_df.head()

In [ ]:
def parse_inc_angle(series):
    # Convert inc_angle to float and replace 'na' with NaN.
    return pd.to_numeric(series.replace("na", np.nan), errors="coerce")

train_df["inc_angle_clean"] = parse_inc_angle(train_df["inc_angle"])
test_df["inc_angle_clean"] = parse_inc_angle(test_df["inc_angle"])

# Fill missing train angles using train median only.
angle_median = train_df["inc_angle_clean"].median()
train_df["inc_angle_clean"] = train_df["inc_angle_clean"].fillna(angle_median)
test_df["inc_angle_clean"] = test_df["inc_angle_clean"].fillna(angle_median)

scaler = StandardScaler()
train_df["inc_angle_scaled"] = scaler.fit_transform(train_df[["inc_angle_clean"]])
test_df["inc_angle_scaled"] = scaler.transform(test_df[["inc_angle_clean"]])

print("Missing train angles after fill:", train_df["inc_angle_clean"].isna().sum())
print("Train label balance:")
print(train_df["is_iceberg"].value_counts(normalize=True))